# Miniproject template

The notebook presents ...


You can modify this notebook as you want for your miniproject, using Markdown cells to add text or images describing your project, and code cells to implement you simulation. You can always have access to the [original version of this document online](https://github.com/flowersteam/vivarium/tree/upf2026/notebooks/sessions/miniproject_template.ipynb).

If you experience trouble during your miniproject, feel free to contact us on Aula Global with a precise description of your problem and a copy of your notebook file. The notebook file is is named `miniproject_template.ipynb` and is located in the folder indicated when you execute the following cell:

In [ ]:
pwd

As usual, let's connect this notebook to the simulator:

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="miniproject")

## Generic scene

We provide a generic scene that you can use for your miniproject, with the specifications that described below.

## Available entities

The scene contains **12 agents** (blue squares) and **32 objects** (green circles). You can freely customize these 44 entities in order to implement you own scenario, as explained below.

## Defining custom subtypes

During the practical sessions, we were using *subtype labels* to distinguish between different categories of entities. The subtype labels were predefined for the purpose of the sessions and we used meaningful labels, such as `"obstacle"` for entities that agents need to avoid or `"resource"` for entities that agents want to forage. These subtype labels were used for several purposes, e.g. enabling agents to selective sense specific entities with commands such as `agent.proximeters(sensed_entities=["obstacle"])`, or specifying which entites to spawn in the environment with command such as `controller.spawn.subtype = "resource"` (see [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb) and [session 4](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_4.ipynb)).

In your miniproject, however, you will want to define your own meaningful subtypes for categorizing the agents and objects according to the scenario you have in mind. Some of you might want to use subtypes for referring to e.g. `"prey"` or  `"predator"` agents, while others might want to instead refer to `"bee"` agents and `"flower"` objects, or whatever you might have in mind. This is the reason why we use here generic subtypes, "`subtype_0"` to `"subtype_7"`, such that you can freely assign any meaning to these labels.

By defaut, the current scene provides 8 generic subtypes. The names of these generic subtypes can be accessed with

In [ ]:
controller.subtypes

As an example, let's imagine you want to implement a classical scenario with prey and predators agents, as well as resource, obstacle and tree objects. For such a scenario you can define you own meaningful subtype labels with:

Note that the choice of the name of the subtype labels is totally up to you, you can choose whatever labels that make sense for you scenario, e.g. `"box"`, `"home"`, `"bird"`, `"flower"`, `"bee"`, whatever. The only limitation is that **you can only define a maximum of 8 subtype labels**. If you provide more than 8 subtypes in the command above, it will raise an error. 

Now that we have defined our example custom subtypes above with `controller.set_subtype_labels(['prey', 'predator', 'resource', 'obstacle', 'tree'])`, the list of available subtypes as been updated accordingly.  We can check it with

In [ ]:
controller.subtypes

The custom subtypes we have define appear at the top of the list. Note that the remaining subtypes at the end of the list still correspond to the original generic subtypes (`"subtype_5"`, `"subtype_6"` and `"subtype_7"`. This is fine, but you will most likely prefer to only refer to your own custom subtypes in your code (as we will do in the examples below).

We strongly recommend to define your custom subtype at the start of your notebook, just after having connected the notebook to the simulation with `controller = VivariumController.start_session(scene_name="miniproject")`. **Define your custom subtypes only once in your notebook and do not change them later in your following code**.

In our miniproject, the generic subtypes will have the following meaning:
- "subtype_0" will refer to prey agents.
- "subtype_1" will refer to predator agents.
- "subtype_2" will refer to trees.
- "subtype_3" will refer to rocks.

Otherwise, if you prefer to directly use meaningful subtype labels in your code, you can follow the [instructions to customize subtype labels](https://github.com/flowersteam/vivarium/tree/upf2026/notebooks/sessions/custom_subtypes.md). This is not mandatory, so only do it if you feel confortable with these instructions. All the examples we provide in the following of this notebook are using the generic subtypes `"subtype_0"` to `"subtype_7"`. If you decide to change them, you will of course have to adapt your code in consequence.

As an example, let's say we would like to change the diameter and color of the entities to the following:

- Among the 12 agents:
    - 8 are considered as *prey* agents. They are assigned with the subtype `"agent_subtype_1", a diameter of 4, the color blue, and a maximum speed of 2.
    - 4 are considered as *predator* agents. They are assigned with the subtype `"agent_subtype_2", a diameter of 6, the color red, and a maximum speed of 1.
- Among the 32 object
    -  24 are considered as *resources*. They are assigned with the subtype `"object_subtype_1", a diameter of 3 and the color green.
    -  8 are consided ad *obstacles¨. They are assigned with the subtype `"object_subtype_1", a diameter of 8 and the color orange.
 
To set up the scene outlines above we can write:

In [ ]:
# Defining prey agents
for agent in controller.agents[0:8]:
    agent.subtype = "prey"
    agent.diameter = 4
    agent.color = "blue"
    agent.max_speed = 2


# Defining predator agents
for agent in controller.agents[8:12]:
    agent.subtype = "predator"
    agent.diameter = 6
    agent.color = "red"
    agent.max_speed = 1


# Defining resource objects
for obj in controller.objects[0:24]:
    obj.subtype = "resource"
    obj.diameter = 3
    obj.color = "green"


# Defining obstacle objects
for obj in controller.objects[24:31]:
    obj.subtype = "obstacle"
    obj.diameter = 8
    obj.color = "orange"

# Defining the tree object
for obj in controller.objects[31:32]:
    obj.subtype = "tree"
    obj.diameter = 16
    obj.color = "brown"

### Attaching behaviors to specific agent's subtypes

In [ ]:
def obstacle_avoidance(agent):
    left, right = agent.proximeters(sensed_entities=["obstacle", "tree"])
    left_motor = 1 - right
    right_motor = 1 - left
    return left_motor, right_motor  


def foraging(agent):
    left, right = agent.proximeters(sensed_entities=["resource"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor


def attack(agent):
    left, right = agent.proximeters(sensed_entities=["prey"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor


def fear(agent):
    left, right = agent.proximeters(sensed_entities=["predator"])
    left_motor = left
    right_motor = right
    return left_motor, right_motor

In [ ]:
# Detach potential previous behaviors from all agents
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)


# Attach behaviors to prey agents
for agent in controller.agents:
    if agent.subtype == "prey":
        agent.attach_behavior(obstacle_avoidance)
        agent.attach_behavior(foraging)
        agent.attach_behavior(fear)


# Attach behaviors to predator agents
for agent in controller.agents:
    if agent.subtype == "predator":
        agent.attach_behavior(obstacle_avoidance)
        agent.attach_behavior(attack)

### Modifying the attributes of entities

In order to assign different subtypes to different entities (either agents or objects), you can use a `for` loop and the python indexing system we have started to see during the practical sessions and that we precise below. 

All agents of the scene are accessible through the `controller.agents` list, which contains the 12 agents available in these scene (the blue squares on the map) ; and all objects through the `controller.objects` list, which contains the 32 objects available in this scene (the green circles on the map). To access multiple elements in these list you can use the bracket indexing notation of Python. For instance, the 3 first agents in the `controller.agents` list are accessed with:

In [ ]:
controller.agents[0:3]

The code cell above accesses agents from the `controller.agents` list with indexes from 0 (included) to 3 (excluded), therefore the three first agents in the list (separated by comma). The "start" and "stop" indexes are indicated within the square bracket and separed by a column, i.e. `[0:3]` above. In Python, the convention is that the stop index is excluded from the list. Therefore, `controller.agents[0:3]` refers to the three first agents of the list, i.e. `controller.agents[0]`, `controller.agents[1]` and `controller.agents[2]`. The same applies to the `controller.objects` list.

Let's check what are the current diameters (i.e. size) and color of all agents. Since the `controller.agents` list contains all agents, we can check it with:


In [ ]:
# Print the current diameter and color of all agents
for agent in controller.agents:
    print(agent.diameter, agent.color)

What the cell above prints is the diameter and color of each agent (one agent per row). As we can see, all agents have a diameter of 4 and the color blue (as we can also visually observe it on the scene map).

Now, as an example, let's say we would like to change the diameter and color of our 12 agents to be the following:

- 8 agents have a diameter of 6 and the color yellow.
- 4 agents have a diameter of 8 and the color red.

To do this we can write:

In [ ]:
# Iterate over the 8 first agents of the list, i.e. controller.agents[0:8]
# and set their diameter to 6 and their color to yellow
for agent in controller.agents[0:8]:
    agent.diameter = 6
    agent.color = "yellow"

# Iterate over the 4 next agents of the list, i.e. controller.agents[8:12]
# and set their diameter to 8 and their color to red
for agent in controller.agents[8:12]:
    agent.diameter = 8
    agent.color = "red"


After having executed the cell above, you will observe on the scene map that the diameter and color of the agens as changed as we wanted. Now let's imagine we want to change the diameter and color of the 32 objects such that:

- 10 objects are blue with a diameter of 7.
- 1 object is orange with a diameter of 14.
- 16 objects are grey with a diameter of 3.
- All the remaining object (i.e. 5 objects, since there are 32 in total) are left unchanged (i.e. the same as the current green objects we see in the scene map).

For this we can use similar code as above, except that we iterate on the `controller.objects` list instead of the `controller.agents` one as above:

In [ ]:
# Iterate over the 8 first agents of the list, i.e. controller.agents[0:8]
# and set their diameter to 6 and their color to yellow
for obj in controller.objects[0:10]:
    obj.diameter = 7
    obj.color = "blue"

# Iterate over the 4 next agents of the list, i.e. controller.agents[8:12]
# and set their diameter to 8 and their color to red
for obj in controller.objects[10:11]:
    obj.diameter = 14
    obj.color = "orange"

# Iterate over the 4 next agents of the list, i.e. controller.agents[8:12]
# and set their diameter to 8 and their color to red
for obj in controller.objects[11:27]:
    obj.diameter = 3
    obj.color = "grey"


# Since we only modify attributes of 27 objects in the code above (from index 0 included to index 27 excluded)
# and that there are 32 objects in total in the scene, the 5 remaining one are left unchanged.

In your miniproject, you can use similar instructions to define different populations of agents or objects and change their attribute as you want, e.g. to make them easy to distinguish or to convey some meaning (for example deciding that an object will represent a tree and will be brown, with a larger diameter than a flower object which will be yellow).

In [ ]:
agent.subtype

In [ ]:
from collections.abc import Iterable
for name, ctrl in controller.controllers.items():
    print(f'CONTROLLER {name} of type {type(ctrl)}')
    print(ctrl.__dict__.keys())
    if '_subtype_labels' in ctrl.__dict__.keys():
        print(ctrl._subtype_labels)
    if isinstance(ctrl, Iterable):
        print(f'\t {ctrl[0].__dict__.keys()}')
    print('============================')

In [ ]:
controller.agents.subtype_labels == controller.agents[0]._subtype_labels

In [ ]:
controller.agents.subtype_labels

In [ ]:
id(controller.agents[0]._subtype_labels)

In [ ]:
controller.simulator.subtype_labels = ['a', 'b', 'c']

In [ ]:
id(controller.simulator.subtype_labels)

### Launching mutliple consumption mechanisms

The scene provides **4 independent *slots* for the consumption mechanim**. Below we use to of them, `slot_1` and `slot_2` as an example where:

- Preys consume resources
- Predators consume preys

In [ ]:
# Preys consume resources
controller.consumption.slot_1.source_subtype = "prey"
controller.consumption.slot_1.target_subtype = "resource"
controller.consumption.slot_1.start = True

In [ ]:
# Predators consume preys
controller.consumption.slot_2.source_subtype = "predator"
controller.consumption.slot_2.target_subtype = "prey"
controller.consumption.slot_2.start = True

### Launching mutliple spawning mechanisms

The scene provides **4 independent *slots* for the spawning mechanim**. Below we use to of them, `slot_1` and `slot_2` as an example where:

- New resources spawn every 100 time steps
- New preys spawn every 200 time steps

In [ ]:
# Spawn resources
controller.spawn.slot_1.subtype = "resource"
controller.spawn.slot_1.period = 100
controller.spawn.slot_1.start = True

In [ ]:
# Spawn preys
controller.spawn.slot_2.subtype = "prey"
controller.spawn.slot_2.period = 200
controller.spawn.slot_2.start = True

In [ ]:
for obj in controller.objects:
    if obj.subtype == "tree":
        obj.mass = 1000
        obj.friction = 1000

## Avalaible entity attributes

You can freely customize the attributes of the entities (either agents or objects).

### All entities

x_position, y_position, orientation, subtype, diameter, color, friction, mass, exists

### Agents
proxs_dist_max, proxs_cos_min, max_speed, wheel_diameter, visible_wheels, visible_proxs, visible

Note: friction seems to no longer make agent drift (probably to change in motor force, which is now always aligned with the front direction..). But one advantage is that reducing the friction makes them move much faster ..

Note (solved): Current it seems that agents can't consume other agents. Might just be a matter of adding an reproduction component to agents in the config (for their death).
**BUT:**
Does spawning of agent deal correctly with subtype? (as it chooses a random non-existing agent of the list, it might be of the wrong subtype. And also wrong behavior, physical_attributes etc..). This might be hard to solve, maube better indicate that agent spawning is not available, creativity emerges from constrains anyway .. (or just explain the limitation). Might be solved with routines, but it's a workaround. And requires controller routines. 
**ACTUALLY**
Yes, only entities of the indicated spawning subtype do spawn, so normally all good.

## Calibrating the simulator speed

This session's environment contains more entities than in the previous sessions, which might slow down the simulation. Let's attach the `obstacle_avoidance` behavior to all agents so that you can observe if it runs fast enough:

In [ ]:
for agent in controller.agents:
    agent.attach_behavior(obstacle_avoidance)

In case you find the agents are moving too slow, you can increase the number of steps the simulation performs on the server for each step performed in this notebook's controller by mofifying the `controller.simulator.env.num_scan_steps` parameter. It is set to 1 by default. If you double it to 2, your simulation will run approximately twice faster:

In [ ]:
controller.simulator.env.num_scan_steps = 2

Use this mechanism wisely, as increasing this number too high will make your agent's behaviors less reactive, in the sense that the time between the proximeter sensing and the motor activations will be longer. We recommend to not increase it above 8 maximum. Only use integer numbers for this parameter. Once you have find a number that suits you, you can detach the obstacle_avoidance behavior with:

In [ ]:
for agent in controller.agents:
    agent.detach_behavior(obstacle_avoidance, stop_motors=True)